关于assignment2的部分，主要是是实现Batch Normalization，Dropout，Convolutional Neural Network，学习Pytorch的使用，和应用Rnns

## Batch Normalization  
batch normalization是一种处理输入数据的方法，其使得数据均值为0，方差接近1，从而提升训练效率。
($x_{i}$的维度为（N，D）)  
### batch normalization 正向传播的公式为：
$$
\mu = \frac{1}{N} \sum_{i=1}^N x_i
$$  
$$
\sigma^2 = \frac{1}{N} \sum_{i=1}^N (x_i - \mu)^2
$$
$$
\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}
$$
完成数据归一化
$$
out=\gamma * \hat{x} + \beta
$$
对数据的缩放和平移，这一层使得网络可以自行学习数据的分布，而非严格的均值为0，方差接近1，增强网络灵活性


### batch normalization的反向传播
$$
\frac{\partial L}{\partial \hat{x}} = \frac{\partial L}{\partial out} * \gamma
$$
$$
\frac{\partial L}{\partial \sigma^2} = \sum_{i=1}^N \frac{\partial L}{\partial \hat{x_i}}*(x_i-/mu)*\frac{-1}{2}*(\sigma^2+\epsilon)^{-3/2}
$$
$$
\frac{\partial L}{\partial \mu} = \frac{1}{N} * \sum_{i=1}^N \frac{\partial L}{\partial \hat{x}}* \hat{x} - \frac{\partial L}{\partial \sigma^2}*\frac{\sum_{i=1}^N -2*(x_i-mu)}{N}
$$
$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial \hat{x}}* \frac{1}{\sqrt{\sigma^2+\epsilon}} - \frac{\partial L}{\partial \sigma^2}*\frac{\sum_{i=1}^N (x_i-\mu)}{N} - \frac{\partial L}{\partial \mu}*\frac{1}{N}
$$
$$
\frac{\partial L}{\partial \gamma} = \sum_{i=1}^N \frac{\partial L}{\partial \hat{x_i}}*\hat{x_i}
$$
$$
\frac{\partial L}{\partial \beta} = \sum_{i=1}^N \frac{\partial L}{\partial \hat{x_i}}
$$w



通过BN将数据标准化，将一些离谱的数据分布拉回正常的，易于训练的区间，使得数据更适用于训练。且在应用时输入的数据通过BN，使数据维度和训练时基本一致。使得模型的效果更稳定。

## Dropout  
Dropout 是一种正则化方法，在训练过程中，随机将一部分神经元权重置零，从而避免过拟合。
### Dropout 的前向传播  
$$
h = out \odot mask * \frac{1}{p}
$$
其中out为输入，mask为随机生成的0-1矩阵（为1的元素个数和总元素个数的比为p），p为概率。  
### Dropout 的后向传播
$$
\frac{\partial L}{\partial out} = \frac{\partial L}{\partial h} \odot mask * \frac{1}{p}
$$

## Convolutional Neural Networks  
卷积神经网络的特色在于，在第一层其将一张完整的图拆分成多个小的视图，这样使得cnn能看到各个部位的线条，色块特征。而更深层的网络则去学习这些浅层特征间更抽象，复杂的联系。 
   
学习后第一层的w  
<img src="picture/output.png" alt="Logo" width="400" />  
可以见得网络学习到了不同的色块和线条特征


### Convolutional Layers  

<img src="picture/image3.png" alt="Logo" width="500" />  

cnn中的参数有     
输入图像的形状（$N$,$D_1$,$W_1$,$H_1$）  
$W$的形状（$D_2$,$D_1$,$F$,$F$）   
输出的形状（$N$,$D_2$,$W_2$,$H_2$）  

$W_1$,$H_1$：卷积核的高和宽（如 3x3）。  
$D_1$：输入通道数（如 RGB 图像为 3）。  
$S$：步长（如 1）。  
$P$: 0填充  
$F$:每个卷积核的尺寸  
$W_2$,$H_2$：输出图像的宽和高。  
$D_2$：输出通道数（即卷积核个数）。  
  
$W_2=（W_1+2P-F)/S+1$  
$H_2=（H_1+2P-F)/S+1$  

通常将每块卷积拆分拉直变成 x_col（$N$,$D_1*F*F$,$W_2*H_2$）  
将w展开成 w_ravel（$D_2$,$D_1*F*F$）  
然后将w_ravel@x_col+b    
最后在转化为（$N$,$D_2$,$W_2$,$H_2$）的结果  

## Pooling Layers
池化层保留局部最强的特征，从而减少参数数量，提升计算效率。（因为相近的数据只有微小平移，所以对于图像特征基本没有丢失）同时能有效防止过拟合
  
<img src="picture/image4.png" alt="Logo" width="500" />   
    
池化层的参数有    
输入图像的形状（$N$,$D_1$,$W_1$,$H_1$）  
输出图像的形状（$N$,$D_1$,$W_2$,$H_2$）  
    
$F$:每个池化核的尺寸   
$S$:步长（如 2）。   
   
$W_2=（W_1-F)/S+1$    
$H_2=（H_1-F)/S+1$    

## pytorch的使用
### Barebone
在使用pytorch的计算时，我们只需要完成前向传播的编写，然后pytorch会自动完成反向传播。
```python
def train_part2(model_fn, params, learning_rate):
    for t, (x, y) in enumerate(loader_train):
        x = x.to(device=device, dtype=dtype)
        y = y.to(device=device, dtype=torch.long)#获取数据并转到对应设备上（一般是GPU）

        scores = model_fn(x, params)#model_fn是用pytorch写的前向传播模型
        loss = F.cross_entropy(scores, y)#使用交叉熵为损失函数

    ###以上步骤可以通过nn.Module来简化实现###

        loss.backward()#自动完成前面全部的反向传播

        with torch.no_grad():#临时关闭自动求导
            for w in params:
                w -= learning_rate * w.grad#通过w.grad调取dw完成w的更新
                w.grad.zero_()

    ###以上步骤可以通过pytorcn.optim来简化实现###
        if t % print_every == 0:
            print('Iteration %d, loss = %.4f' % (t, loss.item()))
            check_accuracy_part2(loader_val, model_fn, params)
            print()
```


### nn.Module
一种更加便捷的实现模型构建的方法。nn.Module是一个抽象类，我们需要继承这个类，来实现我们的model。同时在nn.Module的派生类中，我们可以嵌套使用nn.Module的派生类，实现更复杂的模型。  
在定义模型时，继承nn.Module类，并实现__init__()和forward()方法。  __init__()方法中定义了模型的参数，forward()方法中定义了前向传播的过程。  
```python
import torch.nn as nn
import torch.nn.functional as F

class Model(nn.Module):
    def __init__(self) -> None:#在__init__()方法中定义需要使用的模型
        super().__init__()
        self.conv1 = nn.Conv2d(1, 20, 5)
        self.conv2 = nn.Conv2d(20, 20, 5)

    def forward(self, x):#在forward()方法中定义前向传播过程
        x = F.relu(self.conv1(x))
        return F.relu(self.conv2(x))
```
通过nn.Module类，我们可以定义更复杂的模型。然后我们再使用pytorch.optim来实现对模型参数的优化。

```python
import torch.optim as optim
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
optimizer = optim.Adam([var1, var2], lr=0.0001)

#也可以传入一个可迭代的数组，来制定设置不同参数（无特殊说明则使用默认方法）
optim.SGD([
    {'params': model.base.parameters(), 'lr': 1e-2},
    {'params': model.classifier.parameters()}
], lr=1e-3, momentum=0.9)
```
最后使用step()方法来更新参数
```python
def train(model, optimizer, train_loader, val_loader, epochs):
```